1. Imports & Configuration

In [22]:
import requests

try:
    r = requests.get("http://localhost:11434")
    print("✅ Ollama server is running")
    print("Status code:", r.status_code)
    print(r.text[:200])
except Exception as e:
    print("❌ Ollama server is NOT running or not reachable")
    print(e)

❌ Ollama server is NOT running or not reachable
HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: / (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x00000236583D0190>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))


In [23]:
!ollama --version

'ollama' is not recognized as an internal or external command,
operable program or batch file.


In [24]:
!ollama pull nomic-embed-text
!ollama pull gemma2:2b

'ollama' is not recognized as an internal or external command,
operable program or batch file.
'ollama' is not recognized as an internal or external command,
operable program or batch file.


In [17]:
import os
import time
import hashlib
from pathlib import Path

import ollama
import chromadb
from chromadb.config import Settings

In [18]:
CONFIG = {
    # Models
    "llm_model": "gemma2:2b",
    "embed_model": "nomic-embed-text",

    # Paths
    "documents_dir":   r"C:\1 Raji Sivani\UTD\Sem 4\Agentic AI\Project 1\Lion_kings_RAG_Midterm_Project\Lion_kings_RAG_Midterm_Project\documents",
    "chroma_db_path":  r"C:\1 Raji Sivani\UTD\Sem 4\Agentic AI\Project 1\Lion_kings_RAG_Midterm_Project\Lion_kings_RAG_Midterm_Project\documents",
    "collection_name": "national_parks_v2",

    # Chunking
    "chunk_size": 120,
    "chunk_overlap": 20,

    # Retrieval
    "top_k": 4,
    "candidate_k": 8,
    "min_similarity": 0.62,

    # Conversation
    "max_history": 6,
}

print("✅ Imports successful!")
print(f"   LLM Model    : {CONFIG['llm_model']}")
print(f"   Embed Model  : {CONFIG['embed_model']}")
print(f"   Documents    : {CONFIG['documents_dir']}")
print(f"   Database     : {CONFIG['chroma_db_path']}")

✅ Imports successful!
   LLM Model    : gemma2:2b
   Embed Model  : nomic-embed-text
   Documents    : C:\1 Raji Sivani\UTD\Sem 4\Agentic AI\Project 1\Lion_kings_RAG_Midterm_Project\Lion_kings_RAG_Midterm_Project\documents
   Database     : C:\1 Raji Sivani\UTD\Sem 4\Agentic AI\Project 1\Lion_kings_RAG_Midterm_Project\Lion_kings_RAG_Midterm_Project\documents


2. Document Loader

In [19]:
def load_documents(directory: str) -> list:
    """
    Reads all .txt files from the given folder.
    Returns a list of dicts — one dict per document.
    """
    documents = []
    path = Path(directory)

    # Check if folder exists
    if not path.exists():
        print(f"❌ Folder not found: {directory}")
        return documents

    # Get all .txt files
    files = sorted(path.glob("*.txt"))

    if not files:
        print(f"⚠️  No .txt files found in: {directory}")
        return documents

    print(f"📂 Found {len(files)} files. Loading...")

    for file in files:
        try:
            content = file.read_text(encoding="utf-8").strip()

            if not content:
                print(f"   ⚠️  Skipped (empty): {file.name}")
                continue
            park_name = file.stem.replace("_", " ").title()

            documents.append({
                "filename": file.name,
                "park_name": park_name,
                "content": content,
            })
        
            print(f"   ✅ {file.name}")

        except Exception as e:
            print(f"   ❌ Could not read {file.name}: {e}")

    print(f"\n📄 Total loaded: {len(documents)} documents")
    return documents

documents = load_documents(CONFIG["documents_dir"])

📂 Found 63 files. Loading...
   ✅ acadia.txt
   ✅ american_samoa.txt
   ✅ arches.txt
   ✅ badlands.txt
   ✅ big_bend.txt
   ✅ biscayne.txt
   ✅ black_canyon_gunnison.txt
   ✅ bryce_canyon.txt
   ✅ canyonlands.txt
   ✅ capitol_reef.txt
   ✅ carlsbad_caverns.txt
   ✅ channel_islands.txt
   ✅ congaree.txt
   ✅ crater_lake.txt
   ✅ cuyahoga_valley.txt
   ✅ death_valley.txt
   ✅ denali.txt
   ✅ dry_tortugas.txt
   ✅ everglades.txt
   ✅ gates_of_the_arctic.txt
   ✅ gateway_arch.txt
   ✅ glacier.txt
   ✅ glacier_bay.txt
   ✅ grand_canyon.txt
   ✅ grand_teton.txt
   ✅ great_basin.txt
   ✅ great_sand_dunes.txt
   ✅ great_smoky_mountains.txt
   ✅ guadalupe_mountains.txt
   ✅ haleakala.txt
   ✅ hawaii_volcanoes.txt
   ✅ hot_springs.txt
   ✅ indiana_dunes.txt
   ✅ isle_royale.txt
   ✅ joshua_tree.txt
   ✅ katmai.txt
   ✅ kenai_fjords.txt
   ✅ kings_canyon.txt
   ✅ kobuk_valley.txt
   ✅ lake_clark.txt
   ✅ lassen_volcanic.txt
   ✅ mammoth_cave.txt
   ✅ mesa_verde.txt
   ✅ mount_rainier.txt
   ✅ new

3. Text Chunker

In [20]:
# ── Cell 3: Text Chunker ──────────────────────────────────────────────

def chunk_document(filename: str, park_name: str, content: str, chunk_size: int, overlap: int) -> list:
    """
    Splits one document into overlapping chunks.
    Adds metadata for better filtering later.
    """
    words = content.split()
    chunks = []
    start = 0
    index = 0

    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk_text = " ".join(words[start:end])

        chunk_id = hashlib.md5(f"{filename}_{index}".encode()).hexdigest()[:12]

        chunks.append({
            "id": chunk_id,
            "text": chunk_text,
            "source": filename,
            "park_name": park_name,
            "index": index,
        })

        index += 1
        start += chunk_size - overlap

    return chunks


def chunk_all_documents(documents: list) -> list:
    """
    Chunks every document and combines into one flat list.
    """
    all_chunks = []

    print(f"✂️  Chunking {len(documents)} documents...")
    print(f"   Chunk size: {CONFIG['chunk_size']} words | Overlap: {CONFIG['chunk_overlap']} words\n")

    for doc in documents:
        chunks = chunk_document(
        filename=doc["filename"],
        park_name=doc["park_name"],
        content=doc["content"],
        chunk_size=CONFIG["chunk_size"],
        overlap=CONFIG["chunk_overlap"],
        )
        all_chunks.extend(chunks)
        print(f"   ✅ {doc['filename']:<45} → {len(chunks)} chunks")

    print(f"\n📊 Total chunks created: {len(all_chunks)}")
    return all_chunks


# ── Run it ────────────────────────────────────────────────────────────
all_chunks = chunk_all_documents(documents)

✂️  Chunking 63 documents...
   Chunk size: 120 words | Overlap: 20 words

   ✅ acadia.txt                                    → 11 chunks
   ✅ american_samoa.txt                            → 10 chunks
   ✅ arches.txt                                    → 11 chunks
   ✅ badlands.txt                                  → 10 chunks
   ✅ big_bend.txt                                  → 11 chunks
   ✅ biscayne.txt                                  → 10 chunks
   ✅ black_canyon_gunnison.txt                     → 10 chunks
   ✅ bryce_canyon.txt                              → 11 chunks
   ✅ canyonlands.txt                               → 11 chunks
   ✅ capitol_reef.txt                              → 10 chunks
   ✅ carlsbad_caverns.txt                          → 10 chunks
   ✅ channel_islands.txt                           → 10 chunks
   ✅ congaree.txt                                  → 10 chunks
   ✅ crater_lake.txt                               → 11 chunks
   ✅ cuyahoga_valley.txt                   

4. ChromaDB Indexer

In [21]:
# ── Cell 4: ChromaDB Indexer ──────────────────────────────────────────

def connect_chromadb() -> chromadb.Collection:
    """
    Connects to ChromaDB and returns the collection.
    Creates the chroma_db folder automatically if it doesn't exist.
    """
    client = chromadb.PersistentClient(
        path     = CONFIG["chroma_db_path"],
        settings = Settings(anonymized_telemetry=False),
    )

    collection = client.get_or_create_collection(
        name     = CONFIG["collection_name"],
        metadata = {"hnsw:space": "cosine"},
    )

    print(f"🗄️  ChromaDB connected!")
    print(f"   Collection : {CONFIG['collection_name']}")
    print(f"   Vectors stored so far: {collection.count()}")
    return collection


def embed_and_index(collection: chromadb.Collection, chunks: list) -> None:
    """
    Embeds each chunk using nomic-embed-text
    and stores the vectors in ChromaDB.
    Skips indexing if data already exists.
    """
    # Skip if already indexed
    if collection.count() > 0:
        print(f"✅ Already indexed! {collection.count()} vectors in database.")
        print("   Delete the chroma_db folder and re-run to rebuild.")
        return

    print(f"🔢 Embedding {len(chunks)} chunks...")
    print(f"   Model : {CONFIG['embed_model']}")
    print(f"   This runs ONCE and saves to disk.\n")

    start_time = time.time()

    for i, chunk in enumerate(chunks):
        try:
            # Step 1 — Convert text to vector
            result    = ollama.embeddings(
                model  = CONFIG["embed_model"],
                prompt = chunk["text"],
            )
            embedding = result["embedding"]

            # Step 2 — Store in ChromaDB
            collection.add(
                ids        = [chunk["id"]],
                documents  = [chunk["text"]],
                embeddings = [embedding],
                metadatas = [{
                    "source": chunk["source"],
                    "park_name": chunk["park_name"],
                    "index": chunk["index"]
                }],
            )

            # Progress update every 50 chunks
            if (i + 1) % 50 == 0 or (i + 1) == len(chunks):
                elapsed = round(time.time() - start_time, 1)
                pct     = round((i + 1) / len(chunks) * 100)
                print(f"   [{i+1}/{len(chunks)}] {pct}% done — {elapsed}s elapsed")

        except Exception as e:
            print(f"   ⚠️  Skipped chunk {i} ({chunk['source']}): {e}")
            continue

    total = round(time.time() - start_time, 1)
    print(f"\n✅ Indexing complete!")
    print(f"   Vectors stored : {collection.count()}")
    print(f"   Time taken     : {total}s")


# ── Run it ────────────────────────────────────────────────────────────
collection = connect_chromadb()
embed_and_index(collection, all_chunks)


🗄️  ChromaDB connected!
   Collection : national_parks_v2
   Vectors stored so far: 0
🔢 Embedding 695 chunks...
   Model : nomic-embed-text
   This runs ONCE and saves to disk.

   ⚠️  Skipped chunk 0 (acadia.txt): Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download
   ⚠️  Skipped chunk 1 (acadia.txt): Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download
   ⚠️  Skipped chunk 2 (acadia.txt): Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download
   ⚠️  Skipped chunk 3 (acadia.txt): Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download


KeyboardInterrupt: 

In [ ]:
PARK_NAMES = sorted([
    doc["park_name"].lower() for doc in documents
], key=len, reverse=True)

def detect_park_name(query: str) -> str | None:
    """
    Returns detected park name from the query, if present.
    """
    q = query.lower()
    for park in PARK_NAMES:
        if park in q:
            return park
    return None

In [ ]:
print(f"Total documents loaded : {len(documents)}")
print(f"Total chunks created   : {len(all_chunks)}")
print()

# Show chunks per document
print("Chunks per document:")
print()

from collections import Counter
source_counts = Counter([chunk["source"] for chunk in all_chunks])

for source, count in sorted(source_counts.items()):
    print(f"   {source:<45} → {count} chunks")

5. Retriever

This cell's only job is to take a user's question, convert it to a vector, and find the most similar chunks in ChromaDB.

In [ ]:
# ── Cell 5: Retriever ─────────────────────────────────────────────────

def retrieve(query: str, collection: chromadb.Collection) -> list:
    """
    Park-aware retrieval.
    If a specific park is mentioned, search only that park's chunks.
    Otherwise search across all parks.
    """
    detected_park = detect_park_name(query)

    result = ollama.embeddings(
        model=CONFIG["embed_model"],
        prompt=query,
    )
    query_embedding = result["embedding"]

    query_args = {
        "query_embeddings": [query_embedding],
        "n_results": CONFIG["candidate_k"],
        "include": ["documents", "metadatas", "distances"],
    }

    if detected_park:
        query_args["where"] = {"park_name": detected_park.title()}

    results = collection.query(**query_args)

    chunks = []
    for text, meta, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        similarity = round(1 - distance, 4)

        if similarity >= CONFIG["min_similarity"]:
            chunks.append({
                "text": text,
                "source": meta["source"],
                "park_name": meta["park_name"],
                "similarity": similarity,
            })

    chunks = sorted(chunks, key=lambda x: x["similarity"], reverse=True)
    return chunks[:CONFIG["top_k"]]


# ── Test it ───────────────────────────────────────────────────────────
test_query   = "Which national park has the best wildlife watching?"
test_results = retrieve(test_query, collection)

print(f"🔍 Query: '{test_query}'\n")
for i, chunk in enumerate(test_results, 1):
    print(f"[{i}] Source     : {chunk['source']}")
    print(f"    Similarity : {chunk['similarity']}")
    print(f"    Preview    : {chunk['text'][:120]}...")
    print()
test_query_1 = "where is acadia national park located"
test_results_1 = retrieve(test_query_1, collection)

print(f"Query: '{test_query_1}'\n")
for i, chunk in enumerate(test_results_1, 1):
    print(f"[{i}] Source : {chunk['source']}")
    print(f" Park : {chunk['park_name']}")
    print(f" Similarity : {chunk['similarity']}")
    print(f" Preview : {chunk['text'][:120]}...")
    print()

test_query_2 = "what campgrounds are available in olympic national park"
test_results_2 = retrieve(test_query_2, collection)

print(f"Query: '{test_query_2}'\n")
for i, chunk in enumerate(test_results_2, 1):
    print(f"[{i}] Source : {chunk['source']}")
    print(f" Park : {chunk['park_name']}")
    print(f" Similarity : {chunk['similarity']}")
    print(f" Preview : {chunk['text'][:200]}...")
    print()


6. Generator

In [ ]:
# ── Cell 6: Generator ─────────────────────────────────────────────────

SYSTEM_PROMPT = """
You are a friendly and knowledgeable USA National Parks travel assistant.

You help visitors plan trips by answering questions about national parks.

RULES:
1. Answer only from the provided context.
2. If the user asks about a specific park, answer only about that park.
3. If the answer is not in the context, say: "I don't have that information."
4. Always mention the park name in the answer.
5. Keep the answer clear, practical, and easy to read.
6. Prefer useful travel details such as fees, trails, campgrounds, weather-related planning, and tips when available.
7. Keep the answer under 80 words.
8. Do not guess or combine facts from unrelated parks.
"""


def generate(query: str, chunks: list, chat_history: list) -> tuple:
    """
    Builds prompt from top chunks and returns answer + generation time.
    """
    if not chunks:
        return "I don't have that information.", 0.0

    context_parts = []
    for i, chunk in enumerate(chunks[:CONFIG["top_k"]], 1):
        context_parts.append(
            f"[Source {i}: {chunk['source']} | Park: {chunk['park_name']}]\n{chunk['text']}"
        )

    context = "\n\n---\n\n".join(context_parts)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT}
    ]

    messages.extend(chat_history)

    messages.append({
        "role": "user",
        "content": f"""Use the context below to answer the question.

CONTEXT:
{context}

QUESTION:
{query}
"""
    })

    start = time.time()
    response = ollama.chat(
        model=CONFIG["llm_model"],
        messages=messages,
        options={
            "num_predict": 80,
            "temperature": 0.3,
            "num_ctx": 1536,
            "top_k": 10,
            "top_p": 0.9,
        }
    )
    elapsed = round(time.time() - start, 2)
    answer = response["message"]["content"].strip()

    return answer, elapsed


# ── Test it ───────────────────────────────────────────────────────────
print("🤖 Generating answer...\n")

answer, gen_time = generate(
    query        = test_query,    # from Cell 5
    chunks       = test_results,  # from Cell 5
    chat_history = [],            # empty for now — Cell 7 handles this
)

print(f"Answer:\n{answer}")
print(f"\n⏱️  Generation time: {gen_time}s")

7.Conversation Memory

In [ ]:
# ── Cell 7: Conversation Memory ───────────────────────────────────────

# Simple list that stores the conversation
chat_history = []
conversation_state = {
    "active_park": None
}

def add_to_memory(role: str, content: str) -> None:
    chat_history.append({
        "role": role,
        "content": content,
    })

    max_msgs = CONFIG["max_history"]
    if len(chat_history) > max_msgs:
        del chat_history[:-max_msgs]

def update_active_park(query: str) -> None:
    detected = detect_park_name(query)
    if detected:
        conversation_state["active_park"] = detected.title()

def rewrite_followup_query(query: str) -> str:
    """
    If the user asks a follow-up like 'what is the entry fee?',
    attach the last active park.
    """
    active_park = conversation_state["active_park"]

    if active_park is None:
        return query

    q = query.lower().strip()

    followup_starters = [
        "what about",
        "how about",
        "what is the entry fee",
        "what is the fee",
        "entry fee",
        "camping",
        "trails",
        "weather",
        "best time",
        "can i camp",
    ]

    if detect_park_name(query):
        return query

    if any(q.startswith(x) or q == x for x in followup_starters):
        return f"{query} in {active_park} National Park"

    return query

def clear_memory() -> None:
    chat_history.clear()
    conversation_state["active_park"] = None
    print("🔄 Memory cleared!")

def show_memory() -> None:
    if not chat_history:
        print("(Memory is empty)")
        return

    print(f"📝 Chat History ({len(chat_history)} messages):\n")
    print(f"Active park: {conversation_state['active_park']}\n")

    for i, msg in enumerate(chat_history, 1):
        role = "🧑 User" if msg["role"] == "user" else "🤖 Assistant"
        preview = msg["content"][:80].replace("\n", " ")
        print(f"[{i}] {role}: {preview}...")

8. RAG Pipeline

In [ ]:
# ── Cell 8: RAG Pipeline ──────────────────────────────────────────────

def ask(query: str) -> dict:
    """
    Full RAG pipeline:
    rewrite follow-up -> retrieve -> generate -> update memory
    """
    update_active_park(query)
    rewritten_query = rewrite_followup_query(query)

    t0 = time.time()
    chunks = retrieve(rewritten_query, collection)
    retrieval_time = round(time.time() - t0, 2)

    answer, gen_time = generate(rewritten_query, chunks, chat_history)

    add_to_memory("user", query)
    add_to_memory("assistant", answer)

    return {
        "query": query,
        "rewritten_query": rewritten_query,
        "answer": answer,
        "sources": list(dict.fromkeys([c["source"] for c in chunks])),
        "chunks": chunks,
        "retrieval_time": retrieval_time,
        "generation_time": gen_time,
        "total_time": round(retrieval_time + gen_time, 2),
        "active_park": conversation_state["active_park"],
    }

9.StreamlitUI

In [ ]:
# ── Cell 9: Streamlit UI (REPLACE FULL CELL) ──────────────────────────────
app_code = r'''
import re
import time
import streamlit as st
import ollama
import chromadb
from chromadb.config import Settings

# ── CONFIG ────────────────────────────────────────────────────────────
CONFIG = {
    "llm_model": "qwen2.5:1.5b",
    "embed_model": "nomic-embed-text",
    "chroma_db_path": r"C:\\Users\\14694\\OneDrive - The University of Texas at Dallas\\BUAN 6V99 Agentic AI\\RAG\\chroma_db",
    "collection_name": "national_parks_v2",
    "top_k": 4,
    "max_history": 6,
    "num_predict": 100,
}

SYSTEM_PROMPT = """You are a friendly and knowledgeable USA National Parks travel assistant.
You help visitors plan trips by answering questions about national parks.

RULES:
1. Answer ONLY using the provided context.
2. If the user mentions a specific park by name, answer ONLY about that park.
3. If the context does not contain the answer, say: "I don't have that information."
4. Always mention which park you are talking about.
5. Keep answers clear, helpful, and easy to read.
6. Include practical details like fees, trails, camping, and tips when available.
7. Keep answers under 120 words.
"""

# ── PARK LIST ─────────────────────────────────────────────────────────
PARKS = [
    "acadia", "american samoa", "arches", "badlands", "big bend", "biscayne",
    "black canyon of the gunnison", "bryce canyon", "canyonlands", "capitol reef",
    "carlsbad caverns", "channel islands", "congaree", "crater lake",
    "cuyahoga valley", "death valley", "denali", "dry tortugas", "everglades",
    "gates of the arctic", "gateway arch", "glacier", "glacier bay",
    "grand canyon", "grand teton", "great basin", "great sand dunes",
    "great smoky mountains", "guadalupe mountains", "haleakala",
    "hawaii volcanoes", "hot springs", "indiana dunes", "isle royale",
    "joshua tree", "katmai", "kenai fjords", "kings canyon", "kobuk valley",
    "lake clark", "lassen volcanic", "mammoth cave", "mesa verde",
    "mount rainier", "new river gorge", "north cascades", "olympic",
    "petrified forest", "pinnacles", "redwood", "rocky mountain",
    "saguaro", "sequoia", "shenandoah", "theodore roosevelt",
    "virgin islands", "voyageurs", "white sands", "wind cave",
    "wrangell st elias", "yellowstone", "yosemite", "zion"
]

PARK_FILE_MAP = {
    "acadia": "acadia.txt",
    "american samoa": "american_samoa.txt",
    "arches": "arches.txt",
    "badlands": "badlands.txt",
    "big bend": "big_bend.txt",
    "biscayne": "biscayne.txt",
    "black canyon of the gunnison": "black_canyon_gunnison.txt",
    "bryce canyon": "bryce_canyon.txt",
    "canyonlands": "canyonlands.txt",
    "capitol reef": "capitol_reef.txt",
    "carlsbad caverns": "carlsbad_caverns.txt",
    "channel islands": "channel_islands.txt",
    "congaree": "congaree.txt",
    "crater lake": "crater_lake.txt",
    "cuyahoga valley": "cuyahoga_valley.txt",
    "death valley": "death_valley.txt",
    "denali": "denali.txt",
    "dry tortugas": "dry_tortugas.txt",
    "everglades": "everglades.txt",
    "gates of the arctic": "gates_of_the_arctic.txt",
    "gateway arch": "gateway_arch.txt",
    "glacier": "glacier.txt",
    "glacier bay": "glacier_bay.txt",
    "grand canyon": "grand_canyon.txt",
    "grand teton": "grand_teton.txt",
    "great basin": "great_basin.txt",
    "great sand dunes": "great_sand_dunes.txt",
    "great smoky mountains": "great_smoky_mountains.txt",
    "guadalupe mountains": "guadalupe_mountains.txt",
    "haleakala": "haleakala.txt",
    "hawaii volcanoes": "hawaii_volcanoes.txt",
    "hot springs": "hot_springs.txt",
    "indiana dunes": "indiana_dunes.txt",
    "isle royale": "isle_royale.txt",
    "joshua tree": "joshua_tree.txt",
    "katmai": "katmai.txt",
    "kenai fjords": "kenai_fjords.txt",
    "kings canyon": "kings_canyon.txt",
    "kobuk valley": "kobuk_valley.txt",
    "lake clark": "lake_clark.txt",
    "lassen volcanic": "lassen_volcanic.txt",
    "mammoth cave": "mammoth_cave.txt",
    "mesa verde": "mesa_verde.txt",
    "mount rainier": "mount_rainier.txt",
    "new river gorge": "new_river_gorge.txt",
    "north cascades": "north_cascades.txt",
    "olympic": "olympic.txt",
    "petrified forest": "petrified_forest.txt",
    "pinnacles": "pinnacles.txt",
    "redwood": "redwood.txt",
    "rocky mountain": "rocky_mountain.txt",
    "saguaro": "saguaro.txt",
    "sequoia": "sequoia.txt",
    "shenandoah": "shenandoah.txt",
    "theodore roosevelt": "theodore_roosevelt.txt",
    "virgin islands": "virgin_islands.txt",
    "voyageurs": "voyageurs.txt",
    "white sands": "white_sands.txt",
    "wind cave": "wind_cave.txt",
    "wrangell st elias": "wrangell_st_elias.txt",
    "yellowstone": "yellowstone.txt",
    "yosemite": "yosemite.txt",
    "zion": "zion.txt",
}

# ── CHROMA LOAD ───────────────────────────────────────────────────────
@st.cache_resource
def load_collection():
    client = chromadb.PersistentClient(
        path=CONFIG["chroma_db_path"],
        settings=Settings(anonymized_telemetry=False),
    )
    collection = client.get_or_create_collection(
        name=CONFIG["collection_name"],
        metadata={"hnsw:space": "cosine"},
    )
    return collection

# ── HELPERS ───────────────────────────────────────────────────────────
def detect_park(query: str):
    q = query.lower()
    matches = [park for park in PARKS if park in q]
    if matches:
        matches = sorted(matches, key=len, reverse=True)
        return matches[0]
    return None

def rewrite_query(query: str, active_park: str | None):
    current_park = detect_park(query)
    if current_park:
        return query, current_park

    follow_up_words = [
        "entry fee", "fee", "camping", "campground", "trails", "trail",
        "weather", "best time", "permits", "reservation", "timing",
        "wildlife", "hiking", "lodging", "stay", "visit"
    ]

    q = query.lower()
    is_follow_up = any(word in q for word in follow_up_words)

    if active_park and is_follow_up:
        rewritten = f"For {active_park.title()} National Park: {query}"
        return rewritten, active_park

    return query, active_park

def retrieve(query, collection, active_park=None):
    result = ollama.embeddings(
        model=CONFIG["embed_model"],
        prompt=query,
    )
    query_embedding = result["embedding"]

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=10,
        include=["documents", "metadatas", "distances"],
    )

    chunks = []
    for text, meta, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        source = meta["source"]
        sim = round(1 - distance, 4)

        # boost score if current active park matches source
        if active_park and source == PARK_FILE_MAP.get(active_park):
            sim += 0.12

        chunks.append({
            "text": text,
            "source": source,
            "similarity": round(sim, 4),
        })

    chunks = sorted(chunks, key=lambda x: x["similarity"], reverse=True)

    # if a park is active, prefer that park's file first
    if active_park:
        preferred_source = PARK_FILE_MAP.get(active_park)
        park_chunks = [c for c in chunks if c["source"] == preferred_source]
        other_chunks = [c for c in chunks if c["source"] != preferred_source]

        if park_chunks:
            chunks = park_chunks[:CONFIG["top_k"]]
        else:
            chunks = (chunks[:CONFIG["top_k"]])

    else:
        chunks = chunks[:CONFIG["top_k"]]

    return chunks

def trim_history(chat_history, max_turns=3):
    if not chat_history:
        return []
    return chat_history[-(max_turns * 2):]

def generate(query, chunks, chat_history):
    context_parts = []
    for i, chunk in enumerate(chunks, 1):
        context_parts.append(
            f"[Source {i}: {chunk['source']} | Similarity: {chunk['similarity']}]\n{chunk['text']}"
        )
    context = "\n\n---\n\n".join(context_parts)

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages.extend(trim_history(chat_history, max_turns=2))
    messages.append({
        "role": "user",
        "content": f"""Use the following context to answer the question.

CONTEXT:
{context}

QUESTION:
{query}
"""
    })

    start = time.time()
    response = ollama.chat(
        model=CONFIG["llm_model"],
        messages=messages,
        options={
            "num_predict": CONFIG["num_predict"],
            "temperature": 0.3,
            "num_ctx": 1536,
            "top_k": 10,
            "top_p": 0.9,
        }
    )
    elapsed = round(time.time() - start, 2)
    answer = response["message"]["content"].strip()
    return answer, elapsed

# ── PAGE SETUP ────────────────────────────────────────────────────────
st.set_page_config(
    page_title="National Parks Assistant",
    page_icon="🏞️",
    layout="centered",
)

st.title("🏞️ USA National Parks Assistant")
st.caption("Ask me about the 63 U.S. National Parks")

collection = load_collection()

# ── SESSION STATE ─────────────────────────────────────────────────────
if "messages" not in st.session_state:
    st.session_state.messages = []

if "chat_history" not in st.session_state:
    st.session_state.chat_history = []

if "active_park" not in st.session_state:
    st.session_state.active_park = None

# ── SIDEBAR ───────────────────────────────────────────────────────────
with st.sidebar:
    st.subheader("Current Context")
    st.write("Active park:", st.session_state.active_park.title() if st.session_state.active_park else "None")

    if st.button("Clear Chat"):
        st.session_state.messages = []
        st.session_state.chat_history = []
        st.session_state.active_park = None
        st.rerun()

# ── DISPLAY OLD CHAT ──────────────────────────────────────────────────
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

        if msg["role"] == "assistant" and "sources" in msg:
            with st.expander("Sources used"):
                for src in msg["sources"]:
                    st.write(f"- {src}")

            with st.expander("Top retrieved chunks"):
                for i, ch in enumerate(msg["chunks"], 1):
                    st.markdown(f"**{i}. {ch['source']}**  | similarity: `{ch['similarity']}`")
                    st.write(ch["text"][:350] + "...")

# ── CHAT INPUT ────────────────────────────────────────────────────────
query = st.chat_input("Ask about a national park...")

if query:
    # show user msg
    st.session_state.messages.append({"role": "user", "content": query})
    st.session_state.chat_history.append({"role": "user", "content": query})

    with st.chat_message("user"):
        st.markdown(query)

    with st.chat_message("assistant"):
        with st.spinner("Searching park information..."):
            rewritten_query, detected_or_active = rewrite_query(query, st.session_state.active_park)

            chunks = retrieve(
                query=rewritten_query,
                collection=collection,
                active_park=detected_or_active
            )

            answer, gen_time = generate(
                query=rewritten_query,
                chunks=chunks,
                chat_history=st.session_state.chat_history[:-1]
            )

            retrieval_sources = list(dict.fromkeys([c["source"] for c in chunks]))

            st.markdown(answer)
            st.caption(f"Generation time: {gen_time}s")

            with st.expander("Sources used"):
                for src in retrieval_sources:
                    st.write(f"- {src}")

            with st.expander("Top retrieved chunks"):
                for i, ch in enumerate(chunks, 1):
                    st.markdown(f"**{i}. {ch['source']}**  | similarity: `{ch['similarity']}`")
                    st.write(ch["text"][:350] + "...")

    st.session_state.chat_history.append({"role": "assistant", "content": answer})
    st.session_state.chat_history = st.session_state.chat_history[-CONFIG["max_history"]:]
    st.session_state.active_park = detected_or_active

    st.session_state.messages.append({
        "role": "assistant",
        "content": answer,
        "sources": retrieval_sources,
        "chunks": chunks
    })

# ── FOOTER ────────────────────────────────────────────────────────────
st.divider()
st.caption("RAG assistant with park-aware follow-up handling and source transparency.")
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("✅ New app.py created successfully!")

In [ ]:

# ── Debug Cell ────────────────────────────────────────────
query  = "where is yellowstone national park"
chunks = retrieve(query, collection)

print(f"Query: '{query}'")
print(f"Top {CONFIG['top_k']} chunks retrieved:\n")

for i, chunk in enumerate(chunks, 1):
    print(f"[{i}] Source     : {chunk['source']}")
    print(f"     Similarity : {chunk['similarity']}")
    print(f"     Preview    : {chunk['text'][:100]}...")
    print()


In [ ]:
# ── Cell 11: Quick database check ──────────────────────────────────────
print(f"Total vectors: {collection.count()}")

peek = collection.get(include=["metadatas"])
sources = sorted(set(meta["source"] for meta in peek["metadatas"]))

print(f"Unique parks indexed: {len(sources)}")
print("\nFirst 10 sources:")
for src in sources[:10]:
    print("-", src)

In [ ]:
# ── Cell 12: Retrieval debug test ──────────────────────────────────────
test_query = "what campgrounds are available in olympic national park"

chunks = retrieve(test_query, collection)

print(f"Query: {test_query}\n")
for i, chunk in enumerate(chunks, 1):
    print(f"[{i}] Source     : {chunk['source']}")
    print(f"    Similarity : {chunk['similarity']}")
    print(f"    Preview    : {chunk['text'][:180]}...")
    print()

In [ ]:
# ── Cell 13: Run Streamlit app ─────────────────────────────────────────
!streamlit run app.py